# MobileNetV2 Mixed QAT -> FX -> FPGA Offload

## 0. 환경 및 설정

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.ao.quantization import (
    QuantStub, DeQuantStub,
    get_default_qat_qconfig,
    prepare_qat, disable_observer, convert
)
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torchvision.models.quantization import mobilenet_v2 as qmobilenet_v2
from tqdm import tqdm

In [214]:
# Evaluation
def evaluate_int8(model, dataset, batch_size=32, total_samples=None):
    model.eval()
    correct = 0
    total = 0

    images = []
    labels = []

    with torch.no_grad():
        for sample in tqdm(dataset, total=total_samples, desc="Evaluation", ncols=100):
            images.append(sample["image"])
            labels.append(sample["label"])

            if len(images) == batch_size:
                inputs = torch.stack(images).to("cpu")
                targets = torch.tensor(labels).to("cpu")

                outputs = model(inputs)
                preds = outputs.argmax(dim=1)

                correct += (preds == targets).sum().item()
                total += targets.size(0)

                images.clear()
                labels.clear()

        # leftover
        if len(images) > 0:
            inputs = torch.stack(images).to("cpu")
            targets = torch.tensor(labels).to("cpu")

            outputs = model(inputs)
            preds = outputs.argmax(dim=1)

            correct += (preds == targets).sum().item()
            total += targets.size(0)

    return 100.0 * correct / total

In [215]:
import glob

def iter_subset(dir_path):
    files = sorted(glob.glob(f"{dir_path}/*.pt"))
    for f in files:
        yield torch.load(f)

## 1. 학습된 QAT 모델 로드

In [216]:
# ---------- (1) backbone FP32 ----------
model_fp32 = mobilenet_v2(
    weights=MobileNet_V2_Weights.IMAGENET1K_V1
)
model_fp32.eval()

# ---------- (2) Quantizable backbone ----------
backbone = qmobilenet_v2(quantize=False)
backbone.load_state_dict(model_fp32.state_dict())
backbone.eval()

# ---------- (3) Fuse (⚠️ 필수) ----------
backbone.fuse_model()

In [217]:
# ---------- (4) Mixed PTQ/QAT wrapper ----------
class MobileNetV2_MixedPTQ(torch.nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.stem = backbone.features[0]   # FP32
        self.quant = QuantStub()
        self.features = backbone.features[1:]  # INT8 대상
        self.dequant = DeQuantStub()
        self.classifier = backbone.classifier

    def forward(self, x):
        x = self.stem(x)
        x = self.quant(x)
        x = self.features(x)
        x = self.dequant(x)
        x = F.adaptive_avg_pool2d(x, 1)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [218]:
model_quantizable = MobileNetV2_MixedPTQ(backbone)
model_quantizable.train()

MobileNetV2_MixedPTQ(
  (stem): Conv2dNormActivation(
    (0): ConvReLU2d(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ReLU()
    )
    (1): Identity()
    (2): Identity()
  )
  (quant): QuantStub()
  (features): Sequential(
    (1): QuantizableInvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): ConvReLU2d(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32)
            (1): ReLU()
          )
          (1): Identity()
          (2): Identity()
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1))
        (2): Identity()
      )
      (skip_add): FloatFunctional(
        (activation_post_process): Identity()
      )
    )
    (2): QuantizableInvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): ConvReLU2d(
            (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1))
            (1): ReLU()
       

In [219]:
# ---------- (5) QAT 설정 ----------
torch.backends.quantized.engine = "fbgemm"

model_quantizable.qconfig = get_default_qat_qconfig("fbgemm")
model_quantizable.stem.qconfig = None
model_quantizable.classifier.qconfig = None

In [220]:
# ---------- (6) prepare_qat ----------
model_mixed_qat = prepare_qat(model_quantizable, inplace=False)

# ---------- (7) state_dict 로드 ----------
state_dict = torch.load(
    "model/mobilenetv2_mixed_qat_epoch8_70.12.pth",
    map_location="cpu"
)
model_mixed_qat.load_state_dict(state_dict)

model_mixed_qat.eval()

MobileNetV2_MixedPTQ(
  (stem): Conv2dNormActivation(
    (0): ConvReLU2d(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ReLU()
    )
    (1): Identity()
    (2): Identity()
  )
  (quant): QuantStub(
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([0]), scale=tensor([0.0199]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
      (activation_post_process): MovingAverageMinMaxObserver(min_val=0.0, max_val=2.5325520038604736)
    )
  )
  (features): Sequential(
    (1): QuantizableInvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): ConvReLU2d(
            32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32
            (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
              fake_quant_enabled=tensor([1]), observer

## 2. QAT 모델 INT8 변환 및 검증

In [221]:
# QAT 종류 -> INT8 변환
model_mixed_qat.to("cpu")
model_mixed_qat.apply(disable_observer)

model_mixed_qat_int8 = convert(model_mixed_qat.eval(), inplace=False)

#qat_acc = evaluate_int8(model_mixed_qat_int8, iter_subset("imagenet_subset/val"))
#print(f"Mixed QAT INT8 Accuracy: {qat_acc:.2f}%")

## 3. Project Pointwise Conv FPGA Wrapper 검증

### 3-1. CPU에서 검증

#### 3-1-0. 모델 분석, Wrapper 정의

In [222]:
print(model_mixed_qat_int8)

MobileNetV2_MixedPTQ(
  (stem): Conv2dNormActivation(
    (0): ConvReLU2d(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ReLU()
    )
    (1): Identity()
    (2): Identity()
  )
  (quant): Quantize(scale=tensor([0.0199]), zero_point=tensor([0]), dtype=torch.quint8)
  (features): Sequential(
    (1): QuantizableInvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): QuantizedConvReLU2d(32, 32, kernel_size=(3, 3), stride=(1, 1), scale=0.0885215774178505, zero_point=0, padding=(1, 1), groups=32)
          (1): Identity()
          (2): Identity()
        )
        (1): QuantizedConv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), scale=0.09448595345020294, zero_point=65)
        (2): Identity()
      )
      (skip_add): QFunctional(
        scale=1.0, zero_point=0
        (activation_post_process): Identity()
      )
    )
    (2): QuantizableInvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActiv

In [233]:
# FPGA용 Pointwise Conv 래퍼 정의
class QuantizedConv_FPGA(nn.Module):
    def __init__(self, qconv, with_relu=False):
        super().__init__()
        self.qconv = qconv  # QuantizedConv2d 보관

        # 정적 파라미터 확보 (FPGA에 전달할 값들)
        w_q = qconv.weight()

        self.w_int = w_q.int_repr()                         # int8 weight (shape = [OC, IC, 1, 1])
        self.w_scales = w_q.q_per_channel_scales()          # [OC] shape
        self.w_zps = w_q.q_per_channel_zero_points()        # [OC] shape
        self.w_axis = w_q.q_per_channel_axis()              # usually 0 (out_channels)

        self.bias = qconv.bias()
        self.with_relu = with_relu

        self.y_scale = qconv.scale                          # output scale
        self.y_zero_point = qconv.zero_point                # output zero point

    def forward(self, x):
        # x : quantized input tensor (torch.quint8)
        # TODO : FPGA 호출 코드 작성
        # return y

        # ---------- 1. input quant params ----------
        x_int = x.int_repr().to(torch.int32) # [N, IC, H, W]
        x_scale = x.q_scale()
        x_zp = x.q_zero_point()

        # ---------- 2. weight ----------
        # w_int : [OC, IC, 1, 1] -> [OC, IC]
        w_int = self.w_int.squeeze(-1).squeeze(-1).to(torch.int32)  # [OC, IC]
        w_zps = self.w_zps.to(torch.int32)  # [OC]
        w_scales = self.w_scales            # [OC]

        # ---------- 3. output buffer ----------
        N, IC, H, W = x_int.shape
        OC = w_int.shape[0]

        y_int = torch.zeros((N, OC, H, W), dtype=torch.int32)

        print(f"N : {N}, IC : {IC}, H : {H}, W : {W}, OC : {OC}")

        # ---------- 4. INT32 MAC + bias ----------
        for c in range(OC):
            # (x_int - x_zp): [N, IC, H, W]
            # (w_int[c] - w_zps[c]): [IC]
            acc = (x_int - x_zp) * (w_int[c].view(1, IC, 1, 1) - w_zps[c])
            acc = acc.sum(dim=1)  # [N, H, W], sum over IC

            # ---- bias (INT32 domain) ----
            if self.bias is not None:
                # bias : float
                bias_int = torch.round(
                    self.bias[c] / (x_scale * w_scales[c])
                ).to(torch.int32)
                acc += bias_int

            # ---------- 5. Requantize ----------
            scale = (x_scale * w_scales[c]) / self.y_scale
            acc = torch.round(acc.float() * scale).to(torch.int32)

            # ---------- 6. Output zero-point ----------
            acc += self.y_zero_point

            # ---------- 7. Calmp to uint8 ----------
            if self.with_relu:
                # ReLU 적용
                acc = torch.clamp(acc, self.y_zero_point, 255)
            else:
                # dtype safety용 saturations
                acc = torch.clamp(acc, 0, 255)

            y_int[:, c, :, :] = acc

        # ---------- 8. Make quantized tensor ----------
        y = torch._make_per_tensor_quantized_tensor(
            y_int.to(torch.uint8),
            scale=self.y_scale,
            zero_point=self.y_zero_point
        )          

        print("QuantizedConv2d_FPGA called")
        return y

In [224]:
# FPGA용 Depthwise Conv 래퍼 정의
class QuantizedDepthwiseConv_FPGA(nn.Module):
    def __init__(self, qconv, with_relu=False):
        super().__init__()
        self.qocnv = qconv
        self.with_relu = with_relu

        w_q = qconv.weight() # shape : [C, 1, kH, kW] (float quantized tensor)
        
        self.w_int = w_q.int_repr().to(torch.int32) # shape : [C, 1, kH, kW]
        self.w_scales = w_q.q_per_channel_scales()  # shape : [C]
        self.w_zps = w_q.q_per_channel_zero_points()# shape : [C]

        self.bias = qconv.bias()

        self.y_scale = qconv.scale
        self.y_zero_point = qconv.zero_point

        self.stride = qconv.stride
        self.padding = qconv.padding
        self.kernel_size = qconv.kernel_size

    def forward(self, x):

        x_int = x.int_repr().to(torch.int32) # [N, C, H, W]
        x_scale = x.q_scale()
        x_zp = x.q_zero_point()

        N, C, H, W = x_int.shape
        kH, kW = self.kernel_size

        # output spaital size
        H_out = (H + 2 * self.padding[0] - kH) // self.stride[0] + 1
        W_out = (W + 2 * self.padding[1] - kW) // self.stride[1] + 1

        # padding
        x_pad = torch.nn.functional.pad(
            x_int,
            (self.padding[1], self.padding[1],
             self.padding[0], self.padding[0]),
             value=x_zp
        )

        y_int = torch.zeros((N, C, H_out, W_out), dtype=torch.int32)

        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    h0 = i * self.stride[0]
                    w0 = j * self.stride[1]

                    patch = x_pad[:, c, h0:h0+kH, w0:w0+kW]  # [N, kH, kW]
                    w = self.w_int[c, 0]

                    acc = (patch - x_zp) * (w - self.w_zps[c])
                    acc = acc.sum(dim=(1,2))  # spatial sum

                    # Bias 추가
                    if self.bias is not None:
                        bias_int = torch.round(
                            self.bias[c] / (x_scale * self.w_scales[c])
                        ).to(torch.int32)
                        acc += bias_int

                    # Requantization
                    scale = (x_scale * self.w_scales[c]) / self.y_scale
                    acc = torch.round(acc.float() * scale).to(torch.int32)
                    acc += self.y_zero_point


                    # ReLU / Saturate
                    if self.with_relu:
                        acc = torch.clamp(acc, self.y_zero_point, 255)
                    else:
                        acc = torch.clamp(acc, 0, 255)

                    y_int[:, c, i, j] = acc

        y = torch._make_per_tensor_quantized_tensor(
            y_int.to(torch.uint8),
            scale=self.y_scale,
            zero_point=self.y_zero_point
        )

        return y

In [225]:
from copy import deepcopy
model_mixed_qat_int8_copy = deepcopy(model_mixed_qat_int8)

#### 3-1-1. QuantizedConvReLU2d (Pointwise Conv)

In [226]:
# 교체 전
print("Before:", model_mixed_qat_int8.features[1].conv[0][0])

# 교체
orig_quantizedconvrelu2d = model_mixed_qat_int8.features[1].conv[0][0]
model_mixed_qat_int8.features[1].conv[0][0] = QuantizedConv_FPGA(orig_quantizedconvrelu2d, with_relu=True)
# 교체 후
print("After:", model_mixed_qat_int8.features[1].conv[0][0])

Before: QuantizedConvReLU2d(16, 96, kernel_size=(1, 1), stride=(1, 1), scale=0.046150870621204376, zero_point=0)
After: QuantizedConv_FPGA(
  (qconv): QuantizedConvReLU2d(16, 96, kernel_size=(1, 1), stride=(1, 1), scale=0.046150870621204376, zero_point=0)
)


In [227]:
# 비교
with torch.no_grad():
    x_fp32 = torch.randn(1, 16, 112, 112)

    x_q = torch.quantize_per_tensor(
        x_fp32,
        scale=orig_quantizedconvrelu2d.scale,
        zero_point=orig_quantizedconvrelu2d.zero_point,
        dtype=torch.quint8
    )

    y_ref = orig_quantizedconvrelu2d(x_q)
    y_fpga = model_mixed_qat_int8.features[1].conv[0][0](x_q)

y_ref_int = y_ref.int_repr().to(torch.int32)
y_fpga_int = y_fpga.int_repr().to(torch.int32)

diff = torch.abs(y_ref_int - y_fpga_int).abs()

print("Max difference:", diff.max())

N : 1, IC : 16, H : 112, W : 112, OC : 96
QuantizedConv2d_FPGA called
Max difference: tensor(1, dtype=torch.int32)


#### 3-1-2. QuantizedConv2d (Pointwise Conv)

In [228]:
# 교체 전
print("Before:", model_mixed_qat_int8.features[0].conv[1])

# 교체
orig_quantizedconv2d = model_mixed_qat_int8.features[0].conv[1]
model_mixed_qat_int8.features[0].conv[1] = QuantizedConv_FPGA(orig_quantizedconv2d, with_relu=False)

# 교체 후
print("After:", model_mixed_qat_int8.features[0].conv[1])

Before: QuantizedConv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), scale=0.09448595345020294, zero_point=65)
After: QuantizedConv_FPGA(
  (qconv): QuantizedConv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), scale=0.09448595345020294, zero_point=65)
)


In [232]:
# 비교
with torch.no_grad():
    x_fp32 = torch.randn(1, 32, 112, 112)

    x_q = torch.quantize_per_tensor(
        x_fp32,
        scale=orig_quantizedconv2d.scale,
        zero_point=orig_quantizedconv2d.zero_point,
        dtype=torch.quint8
    )

    y_ref = orig_quantizedconv2d(x_q)
    y_fpga = model_mixed_qat_int8.features[0].conv[1](x_q)

y_ref_int = y_ref.int_repr().to(torch.int32)
y_fpga_int = y_fpga.int_repr().to(torch.int32)

diff = torch.abs(y_ref_int - y_fpga_int).abs()

print("Max difference:", diff.max())

N : 1, IC : 32, H : 112, W : 112, OC : 16
QuantizedConv2d_FPGA called
Max difference: tensor(1, dtype=torch.int32)


#### 3-1-3. QuaantizedConvReLU2d(Depthwise Conv)

In [230]:
# 교체 전
print("Before:", model_mixed_qat_int8.features[0].conv[0][0])

# 교체
orig_quantizedDepthwiseconv = model_mixed_qat_int8.features[0].conv[0][0]
model_mixed_qat_int8.features[0].conv[0][0] = QuantizedDepthwiseConv_FPGA(orig_quantizedDepthwiseconv, with_relu=True)

# 교체 후
print("After:", model_mixed_qat_int8.features[0].conv[0][0])

Before: QuantizedConvReLU2d(32, 32, kernel_size=(3, 3), stride=(1, 1), scale=0.0885215774178505, zero_point=0, padding=(1, 1), groups=32)
After: QuantizedDepthwiseConv_FPGA(
  (qocnv): QuantizedConvReLU2d(32, 32, kernel_size=(3, 3), stride=(1, 1), scale=0.0885215774178505, zero_point=0, padding=(1, 1), groups=32)
)


In [231]:
# 비교
with torch.no_grad():
    x_fp32 = torch.randn(1, 32, 112, 112)

    x_q = torch.quantize_per_tensor(
        x_fp32,
        scale=orig_quantizedDepthwiseconv.scale,
        zero_point=orig_quantizedDepthwiseconv.zero_point,
        dtype=torch.quint8
    )

    y_ref = orig_quantizedDepthwiseconv(x_q)
    y_fpga = model_mixed_qat_int8.features[0].conv[0][0](x_q)

y_ref_int = y_ref.int_repr().to(torch.int32)
y_fpga_int = y_fpga.int_repr().to(torch.int32)

diff = torch.abs(y_ref_int - y_fpga_int).abs()

print("Max difference:", diff.max())

Max difference: tensor(1, dtype=torch.int32)


### 3-2. FPGA에서 검증

#### 3-2-0. 기본 Parameter 및 함수 설정

In [1]:
import numpy as np
import time

In [2]:
# 기본 설정
from pynq import Overlay, Interrupt, allocate, Clocks

ol = Overlay("./design_top.bit")
top = ol.top_0
dma = ol.axi_dma_0

Clocks.fclk0_mhz = 100
print(Clocks.fclk0_mhz)

100.0


In [3]:
# Variable setting
OPCODE_NOP = 0
OPCODE_PARAM = 1
OPCODE_LDSRAM = 2
OPCODE_STSRAM = 3
OPCODE_EX = 4
OPCODE_WBPSRAM = 5
OPCODE_WBPARAM = 6
OPCODE_WRITESRAM_DMA = 7

PARAM_S = 0
PARAM_S_WH = 1
PARAM_IC = 2
PARAM_IC_WH = 3
PARAM_OC = 4
PARAM_OC_WH = 5
PARAM_TRG = 6
PARAM_BASE_WSRAM = 7
PARAM_BASE_WSRAM_WH = 8

TRG_ISRAM = 0
TRG_WSRAM = 1
TRG_PSRAM = 2
TRG_BSRAM = 3

In [4]:
# Basic function
def dec_to_tc(data, bit=8):
    mask = (1 << bit) - 1
    if data >= 0:
        return data
    else:
        datatc = pow(2,bit-1) + (pow(2,bit-1)+data)
        return int(datatc)
    
def tc_to_dec(data, bit=32):
    if data >= pow(2,bit-1):
        datadec = data-2*pow(2,bit-1)
        return int(datadec)
    else:
        return data
    
def instr_param(opvalid, opcode, param, data):
    _instr = opvalid * pow(2,31)
    _instr += opcode * pow(2,28)
    _instr += param * pow(2,8)
    _instr += data * pow(2,0)
    return int(_instr)    

def instr_data(opvalid, opcode, sel, addr, data):
    _instr = opvalid * pow(2,31)
    _instr += opcode * pow(2,28)
    _instr += sel * pow(2,24)
    _instr += addr * pow(2,8)
    _instr += dec_to_tc(data)
    return int(_instr)

def pl_rst():
    top.mmio.write(offset=0, data=dec_to_tc(data=-1,bit=32))
    top.mmio.write(offset=0, data=0)

def finish_check():
    status = top.mmio.read(offset=4)
    return (status >> 31) & 0x1

def valid_check():
    status = top.mmio.read(offset=4)
    return (status >> 30) & 0x1

def wb_param_and_read(param):
    top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_WBPARAM, param=param, data=0))
    while not valid_check():
        pass
    return top.mmio.read(offset=12)

def wb_psram_and_read(sel, addr):
    top.mmio.write(offset=0, data=instr_data(opvalid=1, opcode=OPCODE_WBPSRAM, sel=sel, addr=addr, data=0))
    while not valid_check():
        pass
    return tc_to_dec(top.mmio.read(offset=12), bit=32)

def weight_load(sel,addr,data):
    while True:
        status = top.mmio.read(4)
        pending = (status) & 0x1
        if pending == 0:
            break
    top.mmio.write(offset=0, data=instr_data(opvalid=1, opcode=OPCODE_LDSRAM, sel=sel, addr=addr, data=data))

In [5]:
# Configuration functions
def configure_S(num_rows):
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_S, data=num_rows % 256))
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_S_WH, data=num_rows >> 8))
    # print(f"Complete GEMM shape configuration: S={num_rows}")

def configure_input_channel(num_cols):
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_IC, data=num_cols % 256))
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_IC_WH, data=int(num_cols >> 8)))
    # print(f"Complete input channel configuration: IC={num_cols}")

def configure_output_channel(oc):
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_OC, data=oc % 256))
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_OC_WH, data=oc >> 8))
    # print(f"Complete output channel configuration: OC={oc}")

def dma_send_activation(X):
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_TRG, data=TRG_ISRAM))
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_WRITESRAM_DMA, param=0, data=0))
    
    NHW, IC = X.shape

    i_buf = allocate(shape=(NHW*IC), dtype=np.int32)
    i_buf[:] = X.T.reshape(-1).astype(np.int32)
    
    # print(f"input:{i_buf}")
    
    dma.sendchannel.transfer(i_buf)
    dma.sendchannel.wait()
    # print(f"Complete activation DMA transfer: shape={X.shape}")

def dma_send_weight_vector(W, base_addr=0):
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM, data=base_addr % 256))
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM_WH, data=int(base_addr >> 8)))

    top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_TRG, data=TRG_WSRAM))
    top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_WRITESRAM_DMA, param=0, data=0))    
    
    OC, IC = W.shape

    w_buf = allocate(shape=(OC*IC), dtype=np.int32)
    w_buf[:] = W.T.reshape(-1).astype(np.int32)

    # print(f"weight:{w_buf}")
    
    dma.sendchannel.transfer(w_buf)
    dma.sendchannel.wait()
    # print(f"Complete weight DMA transfer: shape={W.shape}, base_addr={base_addr}")

def execute_receive_psum_dma(S, OC, base_addr=0):
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM, data=base_addr % 256))
    top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM_WH, data=int(base_addr >> 8)))

    out_buf = allocate(shape=(S*OC), dtype=np.int32)
    dma.recvchannel.transfer(out_buf)

    return out_buf

#### 3-2-1. FPGA 결과 검증

In [6]:
def cpu_pointwise_conv(x_int, w_int):
    # x : [N, IC, H, W]
    # w : [OC, IC]

    N, IC, H, W = x_int.shape
    OC = w_int.shape[0]

    X = x_int.transpose(0,2,3,1).reshape(-1, IC)  # [N*H*W, IC]
    X = X.astype(np.int32)
    w_int = w_int.astype(np.int32)
    Y = X @ w_int.T    # [N*H*W, OC]
    Y = Y.reshape(N, H, W, OC).transpose(0,3,1,2)  # [N, OC, H, W]
    return Y

In [7]:
def cpu_pointwise_conv_tiled(x_int, w_int, tile_S=512, tile_IC=32, tile_OC=16):
    N, IC, H, W = x_int.shape
    OC = w_int.shape[0]
    S = N * H * W

    print(f"S = {S}, IC = {IC}, OC = {OC}")
    
    # [S, IC]
    X = x_int.transpose(0,2,3,1).reshape(S, IC)  

    # output
    y_out = np.zeros((N, OC, H, W), dtype=np.int32)
    Y_flat = y_out.reshape(-1, OC) # y_out과 같은 메모리


    # 1. Spatial Tiling (S)
    for s0 in range(0, S, tile_S):
        cur_S = min(tile_S, S - s0)

        # 2. Output-channel Tiling (OC)
        for oc0 in range(0, OC, tile_OC):
            cur_OC = min(tile_OC, OC - oc0)

            # psum tile (same as FPGA)
            psum_tile = np.zeros((cur_S, cur_OC), dtype=np.int32)

            # 3. Input-channel tiling (IC)
            for ic0 in range(0, IC, tile_IC):
                cur_IC = min(tile_IC, IC - ic0)

                X_tile = X[s0:s0+cur_S, ic0:ic0+cur_IC]  # [cur_S, cur_IC]
                W_tile = w_int[oc0:oc0+cur_OC, ic0:ic0+cur_IC]  # [cur_OC, cur_IC]

                # CPU matmul (partial)
                partial = X_tile @ W_tile.T  # [cur_S, cur_OC]

                # accumulate
                psum_tile += partial.astype(np.int32)

            # Write back
            Y_flat[s0:s0+cur_S, oc0:oc0+cur_OC] = psum_tile

    y_out = y_out.reshape(N, H, W, OC).transpose(0,3,1,2)  # [N, OC, H, W]

    return y_out

In [8]:
def fpga_pointwise_conv(x_int, w_int, tile_S=256, tile_IC=32, tile_OC=16):
    # pl_rst()
    N, IC, H, W = x_int.shape
    OC = w_int.shape[0]
    S = N * H * W

    X = x_int.transpose(0,2,3,1).reshape(S, IC)  # [S, IC]

    # output 전체 버퍼 (CPU 메모리)
    y_out = np.zeros((N, OC, H, W), dtype=np.int32)
    Y_flat = y_out.reshape(-1, OC) # y_out과 같은 메모리

    # Timer
    total_start = time.perf_counter()

    activation_dma_time = 0.0
    weight_dma_time = 0.0
    hw_time = 0.0
    accumulate_time = 0.0


    # 1. Spatial Tiling (S)
    for s0 in range(0, S, tile_S):
        cur_S = min(tile_S, S - s0)

        # 2. Output-Channel tiling (OC)
        for oc0 in range(0, OC, tile_OC):
            cur_OC = min(tile_OC, OC - oc0)

            # PSRAM에 올라감 psum 타일
            psum_tile = np.zeros((cur_S, cur_OC), dtype=np.int32)

            configure_output_channel(cur_OC)

            # 3. Input-Channel Tiling (IC)
            for ic0 in range(0, IC, tile_IC):
                cur_IC = min(tile_IC, IC - ic0)

                # Activation / Weight tile
                X_tile = X[s0:s0+cur_S, ic0:ic0+cur_IC]  # [cur_S, cur_IC]
                W_tile = w_int[oc0:oc0+cur_OC, ic0:ic0+cur_IC]  # [cur_OC, cur_IC]

                # configure hardware
                configure_S(cur_S)
                configure_input_channel(cur_IC)

                # DMA send
                t_act = time.perf_counter()
                dma_send_activation(X_tile)
                activation_dma_time += time.perf_counter() - t_act

                t_w = time.perf_counter()
                dma_send_weight_vector(W_tile.T, base_addr=0)
                weight_dma_time += time.perf_counter() - t_w

                # Prepare Recv
                out_buf = execute_receive_psum_dma(cur_S, cur_OC, base_addr=0)

                # EXEC
                t1 = time.perf_counter()
                top.mmio.write(0, instr_param(opvalid=1, opcode=OPCODE_EX, param=0, data=0))

                dma.recvchannel.wait()

                hw_time += time.perf_counter() - t1

                # accumulate psum tile
                t2 = time.perf_counter()
                partial = out_buf.reshape(cur_OC, cur_S).T.astype(np.int32)  # [cur_S, cur_OC]                
                psum_tile += partial
                accumulate_time += time.perf_counter() - t2

            # 4. Write back (after IC accumulation)
            Y_flat[s0:s0+cur_S, oc0:oc0+cur_OC] = psum_tile

    total_time = time.perf_counter() - total_start

    y_out = y_out.reshape(N, H, W, OC).transpose(0,3,1,2)  # [N, OC, H, W]

    print("\n-------FPGA Timing Report-------")
    print(f"Total Time           : {total_time:.6f} sec")
    print(f"Activation DMA Time  : {activation_dma_time:.6f} sec")
    print(f"Weight DMA Time      : {weight_dma_time:.6f} sec")
    print(f"HW Execution Time    : {hw_time:.6f} sec")
    print(f"Accumulation Time    : {accumulate_time:.6f} sec")
    print("--------------------------------\n")

    return y_out

In [94]:
N, IC, H, W = 1, 2, 2, 2
OC = 3

# input: 전부 1
x_int = np.ones((N, IC, H, W), dtype=np.int32)

for ic in range(IC):
    for h in range(H):
        for w in range(W):
            x_int[0, ic, h, w] = ic*100 + h*10 + w
            
            
# weight: 전부 1
w_int = np.random.randint(0,4, (OC, IC), dtype=np.int32)
# w_int = np.ones((OC, IC), dtype=np.int32)

y_cpu = cpu_pointwise_conv(x_int, w_int)
y_fpga = fpga_pointwise_conv(x_int, w_int)

#print("CPU output:\n", y_cpu)
#print("FPGA output:\n", y_fpga)

print("Max difference:", np.max(np.abs(y_cpu - y_fpga)))


-------FPGA Timing Report-------
Total Time           : 0.007711 sec
Activation DMA Time  : 0.003093 sec
Weight DMA Time      : 0.001680 sec
HW Execution Time    : 0.000778 sec
Accumulation Time    : 0.000543 sec
--------------------------------

Max difference: 0


#### 3-2-2. 결과값 및 시간 측정

In [60]:
# (Layer 1) (H = 112, W = 112, S = 12544, IC = 32, OC = 16)
# tile_S = 3750, tile_IC = 32, tile_OC = 16
x_int = np.random.randint(-128, 127, (1, 32, 112, 112), dtype=np.int32)
w_int = np.random.randint(-128, 127, (16, 32), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=3750, tile_IC=32, tile_OC=16)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.108191 sec

-------FPGA Timing Report-------
Total Time           : 0.106655 sec
Activation DMA Time  : 0.050583 sec
Weight DMA Time      : 0.007279 sec
HW Execution Time    : 0.013121 sec
Accumulation Time    : 0.020208 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.110341 sec


In [61]:
# (Layer 2) (H = 112, W = 112, S = 12544, IC = 16, OC = 96)
# tile_S = 625, tile_IC = 16, tile_OC = 96
x_int = np.random.randint(-128, 127, (1, 16, 112, 112), dtype=np.int32)
w_int = np.random.randint(-128, 127, (96, 16), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=625, tile_IC=16, tile_OC=96)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.306778 sec

-------FPGA Timing Report-------
Total Time           : 0.390452 sec
Activation DMA Time  : 0.054251 sec
Weight DMA Time      : 0.036955 sec
HW Execution Time    : 0.070968 sec
Accumulation Time    : 0.121572 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.394185 sec


In [62]:
# (Layer 3) (H = 56, W = 56, S = 3136, IC = 96, OC = 24)
# tile_S = 2500, tile_IC = 96, tile_OC = 24
x_int = np.random.randint(-128, 127, (1, 96, 56, 56), dtype=np.int32)
w_int = np.random.randint(-128, 127, (24, 96), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=2500, tile_IC=96, tile_OC=24)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.126238 sec

-------FPGA Timing Report-------
Total Time           : 0.064590 sec
Activation DMA Time  : 0.037573 sec
Weight DMA Time      : 0.004248 sec
HW Execution Time    : 0.006873 sec
Accumulation Time    : 0.008262 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.070587 sec


In [63]:
# (Layer 4) (H = 56, W = 56, S = 3136, IC = 24, OC = 144)
# tile_S = 416, tile_IC = 24, tile_OC = 144
x_int = np.random.randint(-128, 127, (1, 24, 56, 56), dtype=np.int32)
w_int = np.random.randint(-128, 127, (144, 24), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=416, tile_IC=24, tile_OC=144)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.145972 sec

-------FPGA Timing Report-------
Total Time           : 0.148598 sec
Activation DMA Time  : 0.021340 sec
Weight DMA Time      : 0.015372 sec
HW Execution Time    : 0.027931 sec
Accumulation Time    : 0.045844 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.152212 sec


In [64]:
# (Layer 5) (H = 56, W = 56, S = 3136, IC = 144, OC = 24)
# tile_S = 1666, tile_IC = 144, tile_OC = 24
x_int = np.random.randint(-128, 127, (1, 144, 56, 56), dtype=np.int32)
w_int = np.random.randint(-128, 127, (24, 144), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=1666, tile_IC=144, tile_OC=24)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.172419 sec

-------FPGA Timing Report-------
Total Time           : 0.084064 sec
Activation DMA Time  : 0.055426 sec
Weight DMA Time      : 0.004690 sec
HW Execution Time    : 0.008028 sec
Accumulation Time    : 0.008020 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.088563 sec


In [95]:
# (Layer 6) (H = 56, W = 56, S = 3136, IC = 24, OC = 144)
# tile_S = 416, tile_IC = 24, tile_OC = 24
x_int = np.random.randint(-128, 127, (1, 24, 56, 56), dtype=np.int32)
w_int = np.random.randint(-128, 127, (144, 24), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=416, tile_IC=24, tile_OC=144)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.147206 sec

-------FPGA Timing Report-------
Total Time           : 0.146756 sec
Activation DMA Time  : 0.020335 sec
Weight DMA Time      : 0.014896 sec
HW Execution Time    : 0.027903 sec
Accumulation Time    : 0.046328 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.150353 sec


In [10]:
# (Layer 7) (H = 28, W = 28, S = 784, IC = 144, OC = 32)
# tile_S = 784, tile_IC = 144, tile_OC = 32
x_int = np.random.randint(-128, 127, (1, 144, 28, 28), dtype=np.int32)
w_int = np.random.randint(-128, 127, (32, 144), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=784, tile_IC=144, tile_OC=32)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.047001 sec

-------FPGA Timing Report-------
Total Time           : 0.023702 sec
Activation DMA Time  : 0.012680 sec
Weight DMA Time      : 0.002225 sec
HW Execution Time    : 0.002954 sec
Accumulation Time    : 0.002706 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.027426 sec


In [67]:
# (Layer 8) (H = 28, W = 28, S = 784, IC = 32, OC = 192)
# tile_S = 312, tile_IC = 32, tile_OC = 192
x_int = np.random.randint(-128, 127, (1, 32, 28, 28), dtype=np.int32)
w_int = np.random.randint(-128, 127, (192, 32), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=312, tile_IC=32, tile_OC=192)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.062432 sec

-------FPGA Timing Report-------
Total Time           : 0.050091 sec
Activation DMA Time  : 0.007557 sec
Weight DMA Time      : 0.006370 sec
HW Execution Time    : 0.010020 sec
Accumulation Time    : 0.015315 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.054209 sec


In [68]:
# (Layer 9) (H = 28, W = 28, S = 784, IC = 192, OC = 32)
# tile_S = 784, tile_IC = 192, tile_OC = 32
x_int = np.random.randint(-128, 127, (1, 192, 28, 28), dtype=np.int32)
w_int = np.random.randint(-128, 127, (32, 192), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=784, tile_IC=192, tile_OC=32)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.076844 sec

-------FPGA Timing Report-------
Total Time           : 0.027708 sec
Activation DMA Time  : 0.015801 sec
Weight DMA Time      : 0.002314 sec
HW Execution Time    : 0.003344 sec
Accumulation Time    : 0.002713 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.031568 sec


In [69]:
# (Layer 10) (H = 28, W = 28, S = 784, IC = 32, OC = 192)
# tile_S = 312, tile_IC = 32, tile_OC = 192
x_int = np.random.randint(-128, 127, (1, 32, 28, 28), dtype=np.int32)
w_int = np.random.randint(-128, 127, (192, 32), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=312, tile_IC=32, tile_OC=192)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.060756 sec

-------FPGA Timing Report-------
Total Time           : 0.051311 sec
Activation DMA Time  : 0.007565 sec
Weight DMA Time      : 0.006180 sec
HW Execution Time    : 0.010021 sec
Accumulation Time    : 0.015865 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.054857 sec


In [70]:
# (Layer 11) (H = 28, W = 28, S = 784, IC = 192, OC = 32)
# tile_S = 784, tile_IC = 192, tile_OC = 32
x_int = np.random.randint(-128, 127, (1, 192, 28, 28), dtype=np.int32)
w_int = np.random.randint(-128, 127, (32, 192), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=784, tile_IC=192, tile_OC=32)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.074038 sec

-------FPGA Timing Report-------
Total Time           : 0.026675 sec
Activation DMA Time  : 0.015259 sec
Weight DMA Time      : 0.002378 sec
HW Execution Time    : 0.003342 sec
Accumulation Time    : 0.002779 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.030612 sec


In [71]:
# (Layer 12) (H = 28, W = 28, S = 784, IC = 32, OC = 192)
# tile_S = 312, tile_IC = 32, tile_OC = 192
x_int = np.random.randint(-128, 127, (1, 32, 28, 28), dtype=np.int32)
w_int = np.random.randint(-128, 127, (192, 32), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=312, tile_IC=32, tile_OC=192)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.059963 sec

-------FPGA Timing Report-------
Total Time           : 0.052524 sec
Activation DMA Time  : 0.007419 sec
Weight DMA Time      : 0.006294 sec
HW Execution Time    : 0.010041 sec
Accumulation Time    : 0.016200 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.056091 sec


In [72]:
# (Layer 13) (H = 14, W = 14, S = 196, IC = 192, OC = 64)
# tile_S = 196, tile_IC = 192, tile_OC = 64
x_int = np.random.randint(-128, 127, (1, 192, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (64, 192), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=196, tile_IC=192, tile_OC=64)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.037150 sec

-------FPGA Timing Report-------
Total Time           : 0.014853 sec
Activation DMA Time  : 0.005331 sec
Weight DMA Time      : 0.002578 sec
HW Execution Time    : 0.002148 sec
Accumulation Time    : 0.001569 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.018665 sec


In [73]:
# (Layer 14) (H = 14, W = 14, S = 196, IC = 64, OC = 384)
# tile_S = 156, tile_IC = 64, tile_OC = 384
x_int = np.random.randint(-128, 127, (1, 64, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (384, 64), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=156, tile_IC=64, tile_OC=384)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.057071 sec

-------FPGA Timing Report-------
Total Time           : 0.033395 sec
Activation DMA Time  : 0.005085 sec
Weight DMA Time      : 0.006831 sec
HW Execution Time    : 0.006551 sec
Accumulation Time    : 0.008367 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.037025 sec


In [74]:
# (Layer 15) (H = 14, W = 14, S = 196, IC = 384, OC = 64)
# tile_S = 196, tile_IC = 384, tile_OC = 64
x_int = np.random.randint(-128, 127, (1, 384, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (64, 384), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=196, tile_IC=384, tile_OC=64)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.062577 sec

-------FPGA Timing Report-------
Total Time           : 0.019642 sec
Activation DMA Time  : 0.008853 sec
Weight DMA Time      : 0.003524 sec
HW Execution Time    : 0.003025 sec
Accumulation Time    : 0.001643 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.023107 sec


In [75]:
# (Layer 16) (H = 14, W = 14, S = 196, IC = 64, OC = 384)
# tile_S = 156, tile_IC = 64, tile_OC = 384
x_int = np.random.randint(-128, 127, (1, 64, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (384, 64), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=156, tile_IC=64, tile_OC=384)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.056931 sec

-------FPGA Timing Report-------
Total Time           : 0.034246 sec
Activation DMA Time  : 0.004738 sec
Weight DMA Time      : 0.006930 sec
HW Execution Time    : 0.006532 sec
Accumulation Time    : 0.008551 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.037819 sec


In [76]:
# (Layer 17) (H = 14, W = 14, S = 196, IC = 384, OC = 64)
# tile_S = 196, tile_IC = 384, tile_OC = 64
x_int = np.random.randint(-128, 127, (1, 384, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (64, 384), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=196, tile_IC=384, tile_OC=64)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.062697 sec

-------FPGA Timing Report-------
Total Time           : 0.020712 sec
Activation DMA Time  : 0.009849 sec
Weight DMA Time      : 0.003618 sec
HW Execution Time    : 0.003006 sec
Accumulation Time    : 0.001607 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.024228 sec


In [77]:
# (Layer 18) (H = 14, W = 14, S = 196, IC = 64, OC = 384)
# tile_S = 156, tile_IC = 64, tile_OC = 384
x_int = np.random.randint(-128, 127, (1, 64, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (384, 64), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=156, tile_IC=64, tile_OC=384)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.056509 sec

-------FPGA Timing Report-------
Total Time           : 0.034041 sec
Activation DMA Time  : 0.004783 sec
Weight DMA Time      : 0.006866 sec
HW Execution Time    : 0.006540 sec
Accumulation Time    : 0.008504 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.037616 sec


In [78]:
# (Layer 19) (H = 14, W = 14, S = 196, IC = 384, OC = 64)
# tile_S = 196, tile_IC = 384, tile_OC = 64
x_int = np.random.randint(-128, 127, (1, 384, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (64, 384), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=196, tile_IC=384, tile_OC=64)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.063379 sec

-------FPGA Timing Report-------
Total Time           : 0.020763 sec
Activation DMA Time  : 0.009867 sec
Weight DMA Time      : 0.003544 sec
HW Execution Time    : 0.003028 sec
Accumulation Time    : 0.001627 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.024211 sec


In [79]:
# (Layer 20) (H = 14, W = 14, S = 196, IC = 64, OC = 384)
# tile_S = 156, tile_IC = 64, tile_OC = 384
x_int = np.random.randint(-128, 127, (1, 64, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (384, 64), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=156, tile_IC=64, tile_OC=384)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.052110 sec

-------FPGA Timing Report-------
Total Time           : 0.035056 sec
Activation DMA Time  : 0.005016 sec
Weight DMA Time      : 0.006875 sec
HW Execution Time    : 0.006578 sec
Accumulation Time    : 0.008556 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.038626 sec


In [80]:
# (Layer 21) (H = 14, W = 14, S = 196, IC = 384, OC = 96)
# tile_S = 156, tile_IC = 384, tile_OC = 96
x_int = np.random.randint(-128, 127, (1, 384, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (96, 384), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=156, tile_IC=384, tile_OC=96)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.091488 sec

-------FPGA Timing Report-------
Total Time           : 0.035292 sec
Activation DMA Time  : 0.012309 sec
Weight DMA Time      : 0.010106 sec
HW Execution Time    : 0.005268 sec
Accumulation Time    : 0.002707 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.038822 sec


In [81]:
# (Layer 22) (H = 14, W = 14, S = 196, IC = 96, OC = 576)
# tile_S = 104, tile_IC = 96, tile_OC = 576
x_int = np.random.randint(-128, 127, (1, 96, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (576, 96), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=104, tile_IC=96, tile_OC=576)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.127029 sec

-------FPGA Timing Report-------
Total Time           : 0.050606 sec
Activation DMA Time  : 0.005834 sec
Weight DMA Time      : 0.012041 sec
HW Execution Time    : 0.010527 sec
Accumulation Time    : 0.011636 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.053664 sec


In [82]:
# (Layer 23) (H = 14, W = 14, S = 196, IC = 576, OC = 96)
# tile_S = 196, tile_IC = 576, tile_OC = 96
x_int = np.random.randint(-128, 127, (1, 576, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (96, 576), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=196, tile_IC=576, tile_OC=96)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.151499 sec

-------FPGA Timing Report-------
Total Time           : 0.029431 sec
Activation DMA Time  : 0.011968 sec
Weight DMA Time      : 0.006138 sec
HW Execution Time    : 0.005449 sec
Accumulation Time    : 0.002206 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.032983 sec


In [83]:
# (Layer 24) (H = 14, W = 14, S = 196, IC = 96, OC = 576)
# tile_S = 104, tile_IC = 96, tile_OC = 576
x_int = np.random.randint(-128, 127, (1, 96, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (576, 96), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=104, tile_IC=96, tile_OC=576)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.128652 sec

-------FPGA Timing Report-------
Total Time           : 0.050740 sec
Activation DMA Time  : 0.005451 sec
Weight DMA Time      : 0.012490 sec
HW Execution Time    : 0.010520 sec
Accumulation Time    : 0.011626 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.054350 sec


In [84]:
# (Layer 25) (H = 14, W = 14, S = 196, IC = 576, OC = 96)
# tile_S = 196, tile_IC = 576, tile_OC = 96
x_int = np.random.randint(-128, 127, (1, 576, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (96, 576), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=196, tile_IC=576, tile_OC=96)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.139508 sec

-------FPGA Timing Report-------
Total Time           : 0.029211 sec
Activation DMA Time  : 0.012222 sec
Weight DMA Time      : 0.006115 sec
HW Execution Time    : 0.005455 sec
Accumulation Time    : 0.002220 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.032763 sec


In [85]:
# (Layer 26) (H = 14, W = 14, S = 196, IC = 96, OC = 576)
# tile_S = 104, tile_IC = 96, tile_OC = 576
x_int = np.random.randint(-128, 127, (1, 96, 14, 14), dtype=np.int32)
w_int = np.random.randint(-128, 127, (576, 96), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=104, tile_IC=96, tile_OC=576)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.140162 sec

-------FPGA Timing Report-------
Total Time           : 0.049057 sec
Activation DMA Time  : 0.005392 sec
Weight DMA Time      : 0.012056 sec
HW Execution Time    : 0.010513 sec
Accumulation Time    : 0.012303 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.052696 sec


In [86]:
# (Layer 27) (H = 7, W = 7, S = 49, IC = 576, OC = 160)
# tile_S = 49, tile_IC = 576, tile_OC = 160
x_int = np.random.randint(-128, 127, (1, 576, 7, 7), dtype=np.int32)
w_int = np.random.randint(-128, 127, (160, 576), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=49, tile_IC=576, tile_OC=160)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.063784 sec

-------FPGA Timing Report-------
Total Time           : 0.020836 sec
Activation DMA Time  : 0.004031 sec
Weight DMA Time      : 0.009629 sec
HW Execution Time    : 0.003387 sec
Accumulation Time    : 0.001229 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.024808 sec


In [87]:
# (Layer 28) (H = 7, W = 7, S = 49, IC = 160, OC = 960)
# tile_S = 49, tile_IC = 160, tile_OC = 750
x_int = np.random.randint(-128, 127, (1, 160, 7, 7), dtype=np.int32)
w_int = np.random.randint(-128, 127, (960, 160), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=49, tile_IC=160, tile_OC=750)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.074066 sec

-------FPGA Timing Report-------
Total Time           : 0.039558 sec
Activation DMA Time  : 0.005062 sec
Weight DMA Time      : 0.016336 sec
HW Execution Time    : 0.007276 sec
Accumulation Time    : 0.005334 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.043172 sec


In [88]:
# (Layer 29) (H = 7, W = 7, S = 49, IC = 960, OC = 160)
# tile_S = 49, tile_IC = 960, tile_OC = 125
x_int = np.random.randint(-128, 127, (1, 960, 7, 7), dtype=np.int32)
w_int = np.random.randint(-128, 127, (160, 960), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=49, tile_IC=960, tile_OC=125)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.107647 sec

-------FPGA Timing Report-------
Total Time           : 0.039852 sec
Activation DMA Time  : 0.011533 sec
Weight DMA Time      : 0.016898 sec
HW Execution Time    : 0.005800 sec
Accumulation Time    : 0.001797 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.043338 sec


In [89]:
# (Layer 30) (H = 7, W = 7, S = 49, IC = 160, OC = 960)
# tile_S = 49, tile_IC = 160, tile_OC = 750
x_int = np.random.randint(-128, 127, (1, 160, 7, 7), dtype=np.int32)
w_int = np.random.randint(-128, 127, (960, 160), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=49, tile_IC=160, tile_OC=750)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.069619 sec

-------FPGA Timing Report-------
Total Time           : 0.039586 sec
Activation DMA Time  : 0.004810 sec
Weight DMA Time      : 0.016401 sec
HW Execution Time    : 0.007276 sec
Accumulation Time    : 0.005557 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.043185 sec


In [90]:
# (Layer 31) (H = 7, W = 7, S = 49, IC = 960, OC = 160)
# tile_S = 49, tile_IC = 960, tile_OC = 125
x_int = np.random.randint(-128, 127, (1, 960, 7, 7), dtype=np.int32)
w_int = np.random.randint(-128, 127, (160, 960), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=49, tile_IC=960, tile_OC=125)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.106435 sec

-------FPGA Timing Report-------
Total Time           : 0.039586 sec
Activation DMA Time  : 0.011171 sec
Weight DMA Time      : 0.016582 sec
HW Execution Time    : 0.005795 sec
Accumulation Time    : 0.001839 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.043087 sec


In [91]:
# (Layer 32) (H = 7, W = 7, S = 49, IC = 160, OC = 960)
# tile_S = 49, tile_IC = 160, tile_OC = 750
x_int = np.random.randint(-128, 127, (1, 160, 7, 7), dtype=np.int32)
w_int = np.random.randint(-128, 127, (960, 160), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=49, tile_IC=160, tile_OC=750)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.069916 sec

-------FPGA Timing Report-------
Total Time           : 0.040332 sec
Activation DMA Time  : 0.004914 sec
Weight DMA Time      : 0.016623 sec
HW Execution Time    : 0.007270 sec
Accumulation Time    : 0.005521 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.043973 sec


In [92]:
# (Layer 33) (H = 7, W = 7, S = 49, IC = 960, OC = 320)
# tile_S = 49, tile_IC = 960, tile_OC = 125
x_int = np.random.randint(-128, 127, (1, 960, 7, 7), dtype=np.int32)
w_int = np.random.randint(-128, 127, (320, 960), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=49, tile_IC=960, tile_OC=125)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.209416 sec

-------FPGA Timing Report-------
Total Time           : 0.070054 sec
Activation DMA Time  : 0.016187 sec
Weight DMA Time      : 0.033354 sec
HW Execution Time    : 0.010816 sec
Accumulation Time    : 0.003064 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.073524 sec


In [93]:
# (Layer 34) (H = 7, W = 7, S = 49, IC = 320, OC = 1280)
# tile_S = 49, tile_IC = 320, tile_OC = 375
x_int = np.random.randint(-128, 127, (1, 320, 7, 7), dtype=np.int32)
w_int = np.random.randint(-128, 127, (1280, 320), dtype=np.int32)

# Reference
start = time.perf_counter()
y_ref = cpu_pointwise_conv(x_int, w_int)
cpu_time = time.perf_counter() - start
print(f"[CPU] total time: {cpu_time:.6f} sec")

# FPGA
start = time.perf_counter()
y_fpga = fpga_pointwise_conv(x_int=x_int, w_int=w_int, tile_S=49, tile_IC=320, tile_OC=375)
fpga_time = time.perf_counter() - start

diff = np.max(np.abs(y_ref - y_fpga))
print("Max difference:", diff)

print(f"[FPGA] end-to-end time: {fpga_time:.6f} sec")

[CPU] total time: 0.255976 sec

-------FPGA Timing Report-------
Total Time           : 0.087581 sec
Activation DMA Time  : 0.011944 sec
Weight DMA Time      : 0.040678 sec
HW Execution Time    : 0.016091 sec
Accumulation Time    : 0.007909 sec
--------------------------------

Max difference: 0
[FPGA] end-to-end time: 0.091179 sec
